# BRB-seq LFC vector analysis

Load the saved freshwater-versus-marine DESeq2 contrasts for gill and liver. Inspect missing and zero LFCs, then calculate vector lengths and angles for every contrast pair within each tissue. No significance filtering is applied.

In [ ]:
library(dplyr)
library(tidyr)

repo_root <- "/scratch/ztobias/Gacu_BC_txmics"
notebook_dir <- file.path(repo_root, "jupyter_notebooks")
gill_results <- readRDS(file.path(notebook_dir, "gill_fw_marine_contrasts.rds"))
liver_results <- readRDS(file.path(notebook_dir, "liver_fw_marine_contrasts.rds"))

In [ ]:
read_lfc_list <- function(results) {
  if (!is.list(results) || is.null(names(results))) stop("Expected a named list of contrast results")

  out <- lapply(names(results), function(contrast_name) {
    result <- as.data.frame(results[[contrast_name]])
    if (!"log2FoldChange" %in% names(result)) stop("Missing log2FoldChange in contrast: ", contrast_name)
    genes <- rownames(result)
    if (is.null(genes) || anyDuplicated(genes)) stop("Missing or duplicated gene IDs in contrast: ", contrast_name)
    data.frame(gene = genes, lfc = as.numeric(result$log2FoldChange))
  })
  names(out) <- names(results)
  out
}

lfc_results <- list(
  gill = read_lfc_list(gill_results),
  liver = read_lfc_list(liver_results)
)
lapply(lfc_results, names)

## Missing and zero LFCs

Exact zero LFCs are retained and counted. Missing or non-finite LFCs are excluded from each pairwise comparison.

In [ ]:
lfc_diagnostics <- bind_rows(lapply(names(lfc_results), function(tissue) {
  bind_rows(lapply(names(lfc_results[[tissue]]), function(contrast_name) {
    x <- lfc_results[[tissue]][[contrast_name]]$lfc
    n_finite <- sum(is.finite(x))
    n_zero <- sum(x == 0, na.rm = TRUE)
    data.frame(
      tissue = tissue, contrast = contrast_name, n_genes = length(x),
      n_missing = sum(is.na(x)),
      n_infinite = sum(!is.na(x) & !is.finite(x)),
      n_finite = n_finite, n_zero = n_zero, n_nonzero = n_finite - n_zero,
      pct_missing = 100 * mean(is.na(x)),
      pct_zero_of_finite = if (n_finite > 0) 100 * n_zero / n_finite else NA_real_
    )
  }))
})) %>% arrange(tissue, contrast)

lfc_diagnostics

## Pairwise vector comparisons

Each pair uses genes with finite LFCs in both contrasts; `n_shared_genes` records the overlap. `pearson_angle_deg` matches the legacy notebook's correlation-based angle. `cosine_angle_deg` is the geometric angle from the uncentered LFC vectors.

In [ ]:
pairwise_vector_stats <- function(contrast_results, tissue) {
  contrast_names <- names(contrast_results)
  if (length(contrast_names) < 2) return(data.frame())

  pair_idx <- combn(seq_along(contrast_names), 2, simplify = FALSE)
  bind_rows(lapply(pair_idx, function(idx) {
    name_1 <- contrast_names[idx[1]]
    name_2 <- contrast_names[idx[2]]
    shared <- inner_join(contrast_results[[name_1]], contrast_results[[name_2]],
                        by = "gene", suffix = c("_1", "_2")) %>%
      filter(is.finite(lfc_1), is.finite(lfc_2))
    x <- shared$lfc_1
    y <- shared$lfc_2
    norm_x <- sqrt(sum(x^2))
    norm_y <- sqrt(sum(y^2))
    cosine <- if (norm_x > 0 && norm_y > 0) sum(x * y) / (norm_x * norm_y) else NA_real_
    correlation <- if (length(x) >= 2 && sd(x) > 0 && sd(y) > 0) cor(x, y) else NA_real_

    data.frame(
      tissue = tissue, contrast_1 = name_1, contrast_2 = name_2,
      n_shared_genes = length(x),
      n_zero_1 = sum(x == 0), n_zero_2 = sum(y == 0),
      n_both_zero = sum(x == 0 & y == 0),
      l2_length_1 = norm_x, l2_length_2 = norm_y,
      pearson_angle_deg = if (!is.na(correlation)) acos(pmax(-1, pmin(1, correlation))) * 180 / pi else NA_real_,
      cosine_angle_deg = if (!is.na(cosine)) acos(pmax(-1, pmin(1, cosine))) * 180 / pi else NA_real_
    )
  }))
}

vector_pairs <- bind_rows(lapply(names(lfc_results), function(tissue) {
  pairwise_vector_stats(lfc_results[[tissue]], tissue)
}))
vector_pairs %>% arrange(tissue, contrast_1, contrast_2)

In [ ]:
vector_pairs %>%
  group_by(tissue) %>%
  summarise(n_contrast_pairs = n(), .groups = "drop")

vector_pairs %>%
  group_by(tissue) %>%
  slice_min(n_shared_genes, n = 10, with_ties = FALSE) %>%
  select(tissue, contrast_1, contrast_2, n_shared_genes, n_zero_1, n_zero_2,
         pearson_angle_deg, cosine_angle_deg)